# 08_stage3_step1_target — 3단계 Step 1: 대상 추출 · 수치 재현

목적: `stage3_step0_rules.md`의 `[ ]` 칸(재현값, 층별 건수, `weight_all` 범위, 3분위 경계)을 채울 숫자를 만든다.

- 이 노트북은 **불만 유형을 보지 않는다.** 코멘트 원문을 출력하지 않고, 길이(글자 수)만 계산한다.
- 샘플링(seed 사용)은 하지 않는다. 층별 건수와 가중치는 **미리보기**일 뿐이고, 실제 추출은 Step 0 커밋 뒤 Step 2에서 한다.
- 실행 경로 기준은 olist 폴더(E-04). 데이터 파일은 `.gitignore` 대상이라 커밋하지 않는다.
- 노트북 안의 "참고값"은 다른 대화에서 나온 **미검증 값**이다. 재현되지 않아도 정상일 수 있고, 안 맞으면 추출 코드부터 의심한다.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

MASTER  = Path("data/processed/master_features.parquet")    # 30컬럼 (master.parquet 26컬럼도 가능)
REVIEWS = Path("data/raw/olist_order_reviews_dataset.csv")
OUT     = Path("data/processed/stage3_target_6400.parquet")  # .gitignore 대상, 커밋 안 함
SAVE    = True

TARGET_N, MIN_PER = 300, 10
checks = []

def check(name, got, expected=None, note=""):
    """expected가 있으면 일치/불일치, 없으면 참고로 기록 (중간에 멈추지 않고 마지막에 한꺼번에 본다)"""
    status = "참고" if expected is None else ("일치" if got == expected else "불일치")
    checks.append({"항목": name, "기대값": expected, "이번 값": got, "상태": status, "비고": note})

if not REVIEWS.exists():
    cand = list(Path("data/raw").rglob("*order_reviews*.csv"))
    assert cand, "원본 reviews CSV 경로를 확인하세요"
    REVIEWS = cand[0]
print("reviews:", REVIEWS)

reviews: data\raw\olist_order_reviews_dataset.csv


## 1. master 불러오기 · 컬럼 확인

In [2]:
master = pd.read_parquet(MASTER)
need = ["order_id", "review_id", "review_score", "low", "late", "review_comment_message",
        "purchase_ts", "freight_sum", "price_sum", "n_items", "n_sellers", "category_en",
        "order_delivered_customer_date", "order_estimated_delivery_date"]
missing = [c for c in need if c not in master.columns]
assert not missing, f"master에 없는 컬럼: {missing}"
print(master.shape)

(96204, 30)


## 2. 대상 추출 (late==0 & low==1 & 코멘트 있음)

In [3]:
has_comment = master["review_comment_message"].notna()
t3 = master[(master["late"] == 0) & (master["low"] == 1) & has_comment].copy()

for c in ["purchase_ts", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    t3[c] = pd.to_datetime(t3[c])

n_blank = int((t3["review_comment_message"].str.strip() == "").sum())
print("대상 주문:", len(t3), "/ 고유 review_id:", t3["review_id"].nunique(), "/ 공백만 있는 코멘트:", n_blank)
check("대상 주문 수", len(t3), 6449, "decisions.md D-00d")
check("고유 review_id 수", t3["review_id"].nunique(), 6400, "decisions.md D-00d")
check("공백만 있는 코멘트(참고)", n_blank, None, "0이 아니면 '코멘트 있음' 정의를 확인")

대상 주문: 6449 / 고유 review_id: 6400 / 공백만 있는 코멘트: 0


## 3. 리뷰 시각 조인 (order_id 기준, 행 수 불변 assert)
dedup 규칙: `review_answer_timestamp DESC → review_creation_date DESC → review_id ASC` 로 order_id당 1건.

In [4]:
rev = pd.read_csv(REVIEWS, parse_dates=["review_creation_date", "review_answer_timestamp"])
rev = (rev.sort_values(["order_id", "review_answer_timestamp", "review_creation_date", "review_id"],
                       ascending=[True, False, False, True])
          .drop_duplicates("order_id", keep="first"))
check("reviews dedup 후 행 수", len(rev), 98673, "decisions.md D-00b")

rev_small = (rev[["order_id", "review_id", "review_answer_timestamp", "review_creation_date"]]
             .rename(columns={"review_id": "review_id_rev"}))
n_before = len(t3)
t3 = t3.merge(rev_small, on="order_id", how="left", validate="m:1")
assert len(t3) == n_before, "조인 후 행 수가 달라졌습니다"
print("조인 후 행 수:", len(t3), "/ 시각 결측:", int(t3["review_answer_timestamp"].isna().sum()),
      "/ master와 review_id 불일치:", int((t3["review_id"] != t3["review_id_rev"]).sum()))

조인 후 행 수: 6449 / 시각 결측: 0 / master와 review_id 불일치: 0


## 4. 파생 열: 도착 전 응답 · 분기 · late 플래그

In [5]:
t3["answered_before_delivery"] = t3["review_answer_timestamp"] < t3["order_delivered_customer_date"]   # 프로젝트 정의(D-00c)
t3["created_before_delivery"]  = t3["review_creation_date"]    < t3["order_delivered_customer_date"]   # 참고용
t3["late_ts"]          = (t3["order_delivered_customer_date"] > t3["order_estimated_delivery_date"]).astype(int)
t3["is_aug_2018"]      = ((t3["purchase_ts"].dt.year == 2018) & (t3["purchase_ts"].dt.month == 8)).astype(int)
t3["is_jul_aug_2018"]  = ((t3["purchase_ts"].dt.year == 2018) & (t3["purchase_ts"].dt.month.isin([7, 8]))).astype(int)
t3["q"] = t3["purchase_ts"].dt.to_period("Q").astype(str)
print(t3[["answered_before_delivery", "created_before_delivery", "late_ts", "is_aug_2018", "is_jul_aug_2018"]].sum())

answered_before_delivery     20
created_before_delivery     210
late_ts                     117
is_aug_2018                 344
is_jul_aug_2018             733
dtype: int64


## 5. review_id 중복 처리 (purchase_ts가 이른 주문 1건)
중복 review_id는 점수·본문·시각이 같다고 알려져 있으니 먼저 확인하고, 동점(같은 purchase_ts)이 있는지도 센다.

In [6]:
dup_mask = t3["review_id"].duplicated(keep=False)
print("중복 review_id에 걸린 주문:", int(dup_mask.sum()), "/ 중복 review_id 수:", t3.loc[dup_mask, "review_id"].nunique())

chk = (t3[dup_mask].groupby("review_id")
       .agg(n_orders=("order_id", "nunique"), n_score=("review_score", "nunique"),
            n_msg=("review_comment_message", "nunique"), n_answer_ts=("review_answer_timestamp", "nunique")))
print("점수 다른 중복:", int((chk.n_score > 1).sum()), "/ 본문 다른 중복:", int((chk.n_msg > 1).sum()),
      "/ 응답시각 다른 중복:", int((chk.n_answer_ts > 1).sum()))

mn = t3.groupby("review_id")["purchase_ts"].transform("min")
n_min = t3[t3["purchase_ts"] == mn].groupby("review_id").size()
n_ties = int((n_min > 1).sum())
print("purchase_ts 동점 review_id 수:", n_ties, "(0이 아니면 Step 0에 동점 규칙을 한 줄 추가: 예) order_id 오름차순)")
check("purchase_ts 동점 review_id 수(참고)", n_ties, None)

t3u = (t3.sort_values(["review_id", "purchase_ts", "order_id"])
         .drop_duplicates("review_id", keep="first")
         .reset_index(drop=True))
check("중복 제거 후 건수", len(t3u), 6400, "decisions.md D-00d")

중복 review_id에 걸린 주문: 97 / 중복 review_id 수: 48
점수 다른 중복: 0 / 본문 다른 중복: 0 / 응답시각 다른 중복: 0
purchase_ts 동점 review_id 수: 18 (0이 아니면 Step 0에 동점 규칙을 한 줄 추가: 예) order_id 오름차순)


## 6. 수치 대조 (6,449주문 / 6,400리뷰 두 단위 모두)
참고값은 다른 대화에서 나온 값(**미검증**)이다. 어느 단위 기준인지도 불분명하므로 두 단위를 나란히 본다.

In [7]:
def summarize(df, label):
    L = df["review_comment_message"].str.len()
    return {"단위": label, "건수": len(df),
            "1점": int((df["review_score"] == 1).sum()), "2점": int((df["review_score"] == 2).sum()),
            "길이 중앙값": float(L.median()), "길이 최대": int(L.max()),
            "도착 전 응답(answer_ts)": int(df["answered_before_delivery"].sum()),
            "도착 전 응답(creation)": int(df["created_before_delivery"].sum()),
            "2017Q1": int((df["q"] == "2017Q1").sum()), "2018Q1": int((df["q"] == "2018Q1").sum())}

ref = {"단위": "참고값(미검증)", "건수": None, "1점": 4816, "2점": 1633, "길이 중앙값": 94.0, "길이 최대": 206,
       "도착 전 응답(answer_ts)": 20, "도착 전 응답(creation)": 27, "2017Q1": 354, "2018Q1": 1486}
summary = pd.DataFrame([summarize(t3, "6,449주문"), summarize(t3u, "6,400리뷰"), ref]).set_index("단위")
display(summary.T)

print("분기별 건수 (6,400리뷰 기준)")
display(t3u["q"].value_counts().sort_index().to_frame("건수").T)
print("late 플래그 (6,400리뷰 기준): late_ts =", int(t3u["late_ts"].sum()),
      "/ 8월(2018) 비율 =", round(t3u["is_aug_2018"].mean() * 100, 1), "%",
      "/ 7~8월 비율 =", round(t3u["is_jul_aug_2018"].mean() * 100, 1), "%")

단위,"6,449주문","6,400리뷰",참고값(미검증)
건수,6449.0,6400.0,NaN
1점,4816.0,4779.0,4816.0
2점,1633.0,1621.0,1633.0
길이 중앙값,94.0,94.0,94.0
길이 최대,206.0,206.0,206.0
도착 전 응답(answer_ts),20.0,15.0,20.0
도착 전 응답(creation),210.0,201.0,27.0
2017Q1,354.0,347.0,354.0
2018Q1,1486.0,1474.0,1486.0


분기별 건수 (6,400리뷰 기준)


q,2017Q1,2017Q2,2017Q3,2017Q4,2018Q1,2018Q2,2018Q3
건수,347,589,739,1167,1474,1353,731


late 플래그 (6,400리뷰 기준): late_ts = 115 / 8월(2018) 비율 = 5.4 % / 7~8월 비율 = 11.4 %


## 7. 층별 건수 · weight_all 미리보기 (추출은 Step 2에서)
Step 2의 할당 규칙과 같다: 비례 + 층당 최소 10건, 층 크기보다 많이 못 뽑음.

In [8]:
t3u["stratum"] = t3u["q"] + "_" + t3u["review_score"].astype(int).astype(str)
N_h = t3u["stratum"].value_counts().sort_index()
prop = np.round(TARGET_N * N_h / N_h.sum())
n_h = np.minimum(np.maximum(MIN_PER, prop), N_h).astype(int)
w = N_h / n_h

plan = pd.DataFrame({"N_h": N_h, "n_h": n_h, "weight_all": w.round(2)})
display(plan)
print("층 수:", len(plan), "/ 총 샘플:", int(n_h.sum()),
      "/ weight_all 범위: %.2f ~ %.2f" % (w.min(), w.max()),
      "/ 최소 10건 보정이 걸린 층:", int((prop < MIN_PER).sum()),
      "/ 층 크기 때문에 n_h가 줄어든 층:", int((N_h < np.maximum(MIN_PER, prop)).sum()))
check("총 샘플 수(미리보기)", int(n_h.sum()), None, "300을 조금 넘을 수 있음 → Step 0에 기록")
check("weight_all 최소", round(float(w.min()), 2), None)
check("weight_all 최대", round(float(w.max()), 2), None)

,N_h,n_h,weight_all
stratum,,,
2017Q1_1,252,12,21.00
2017Q1_2,95,10,9.50
2017Q2_1,439,21,20.90
2017Q2_2,150,10,15.00
2017Q3_1,521,24,21.71
2017Q3_2,218,10,21.80
2017Q4_1,874,41,21.32
2017Q4_2,293,14,20.93
2018Q1_1,1124,53,21.21


층 수: 14 / 총 샘플: 310 / weight_all 범위: 9.50 ~ 22.00 / 최소 10건 보정이 걸린 층: 3 / 층 크기 때문에 n_h가 줄어든 층: 0


## 8. freight_sum 3분위 경계 (대상 6,400건 기준, 라벨링 전에 고정)
경계에 동점이 많으면 분위별 건수가 한쪽으로 쏠린다. 아래 분위별 건수를 보고 규칙 파일의 경계값 칸에 적는다.

In [9]:
fs = t3u["freight_sum"]
print("결측:", int(fs.isna().sum()), "/ 0원:", int((fs == 0).sum()), "/ 고유값:", int(fs.nunique()))
q1, q2 = fs.quantile([1 / 3, 2 / 3])
print("경계: q1 = %.4f, q2 = %.4f" % (q1, q2))
tercile = pd.cut(fs, [-np.inf, q1, q2, np.inf], labels=["하 (<= q1)", "중 (q1 < x <= q2)", "상 (> q2)"])
display(tercile.value_counts().sort_index().to_frame("건수").T)
check("freight_sum 3분위 경계 q1", round(float(q1), 4), None)
check("freight_sum 3분위 경계 q2", round(float(q2), 4), None)

결측: 0 / 0원: 22 / 고유값: 2894
경계: q1 = 15.9400, q2 = 28.1900


freight_sum,하 (<= q1),중 (q1 < x <= q2),상 (> q2)
건수,2137,2130,2133


## 9. 점검 요약 · 저장

In [10]:
report = pd.DataFrame(checks)
display(report)

n_bad = int((report["상태"] == "불일치").sum())
print("불일치 항목:", n_bad, "→", "원인부터 찾기 (Step 0 커밋 전)" if n_bad else "기대값 모두 일치")

if SAVE:
    cols = ["review_id", "order_id", "purchase_ts", "q", "review_score", "review_comment_message", "stratum",
            "review_answer_timestamp", "review_creation_date", "answered_before_delivery",
            "late_ts", "is_aug_2018", "is_jul_aug_2018",
            "freight_sum", "price_sum", "n_items", "n_sellers", "category_en"]
    t3u[cols].to_parquet(OUT, index=False)
    print("저장:", OUT, t3u[cols].shape)

,항목,기대값,이번 값,상태,비고
0,대상 주문 수,6449.0,6449.00,일치,decisions.md D-00d
1,고유 review_id 수,6400.0,6400.00,일치,decisions.md D-00d
2,공백만 있는 코멘트(참고),NaN,0.00,참고,0이 아니면 '코멘트 있음' 정의를 확인
3,reviews dedup 후 행 수,98673.0,98673.00,일치,decisions.md D-00b
4,purchase_ts 동점 review_id 수(참고),NaN,18.00,참고,
5,중복 제거 후 건수,6400.0,6400.00,일치,decisions.md D-00d
6,총 샘플 수(미리보기),NaN,310.00,참고,300을 조금 넘을 수 있음 → Step 0에 기록
7,weight_all 최소,NaN,9.50,참고,
8,weight_all 최대,NaN,22.00,참고,
9,freight_sum 3분위 경계 q1,NaN,15.94,참고,


불일치 항목: 0 → 기대값 모두 일치
저장: data\processed\stage3_target_6400.parquet (6400, 18)
